**XGBoost (Extreme Gradient Boosting)** is an optimized, highly scalable library implementation of Gradient Boosted Decision Trees (GBDT).

# What Problem It Solves
- **Overfitting & Variance:** Traditional GBDT builds trees aggressively based only on residual errors. XGBoost incorporates L1 ($\alpha$) and L2 ($\lambda$) regularization directly into its objective function to control model complexity and penalize overly deep or split-heavy trees.
- **Computational Inefficiency:** Standard GBDTs evaluate every possible feature split sequentially. XGBoost uses parallelized tree building, cache-aware data structures, and quantile-based approximate split algorithms to scale to massive datasets.
- **Missing & Sparse Data:** XGBoost learns a default branch direction for missing or zero values automatically during training rather than requiring manual imputation.

# Core Mechanism

1. **Base Prediction ($F_0$)**

    **Formula**:
    $$F_0(x) = \arg\min_c \sum_{i=1}^N L(y_i, c)$$
    
    - **Regression (MSE):** $F_0(x) = \bar{y} = \frac{1}{N}\sum_{i=1}^N y_i$
    - **Binary Classification (Log-Loss):** $F_0(x) = \ln\left(\frac{\sum y_i}{N - \sum y_i}\right)$

2. **Gradients ($g_i$) and Hessians ($h_i$)**

    **Formulas**
    $$g_i = \frac{\partial L(y_i, \hat{y}_i^{(m-1)})}{\partial \hat{y}_i^{(m-1)}}, \quad h_i = \frac{\partial^2 L(y_i, \hat{y}_i^{(m-1)})}{\partial \left(\hat{y}_i^{(m-1)}\right)^2}$$
    
    - **Regression ($L = \frac{1}{2}(y_i - \hat{y}_i)^2$)**:
    $$g_i = \hat{y}_i - y_i, \quad h_i = 1$$
    
    - **Classification ($L = \text{LogLoss}$)**:
    $$g_i = p_i - y_i, \quad h_i = p_i(1 - p_i)$$

    - $g_i$ tells the tree the direction and magnitude of error. $h_i$ acts as a local curvature scale (confidence weight) that prevents overshooting during optimization.

3. **Node Similarity Score ($Similarity$)**

    **Formula**
    $$\text{Similarity} = \frac{\left( \sum_{i \in I} g_i \right)^2}{\sum_{i \in I} h_i + \lambda}$$

    - It represents that Do the errors in this group agree with each other? or Measures how strongly a group of samples agrees on a single correction direction.

4. **Split Gain ($\text{Gain}$)**

    **Formula**
    $$\text{Gain} = \frac{1}{2} \left[ \frac{\left(\sum_{i \in I_L} g_i\right)^2}{\sum_{i \in I_L} h_i + \lambda} + \frac{\left(\sum_{i \in I_R} g_i\right)^2}{\sum_{i \in I_R} h_i + \lambda} - \frac{\left(\sum_{i \in I} g_i\right)^2}{\sum_{i \in I} h_i + \lambda} \right] - \gamma$$

    - Measures how much overall loss drops if you split a group of samples into two distinct branches.

5. Regularized Leaf Weight ($w_j^*$)

    **Formula**
    $$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$

    - Represents: "The optimal correction step size."

6. **Model Update ($F_m$)**
    
    **Formula**
    $$F_m(x) = F_{m-1}(x) + \nu \cdot \sum_{j=1}^J w_j^* \cdot \mathbb{I}(x \in R_j)$$

    - Rrepresents : "Take a small, safe step forward."


**Final Inference Formula**
$$\hat{y}_{\text{final}} = \text{Transform}\left( F_0(x) + \nu \sum_{m=1}^M f_m(x) \right)$$

# XGBOOST (Regression)

**Dataset**
- $X = [1, 2, 3, 4]$
- $y = [10, 20, 25, 35]$ (Regression using Mean Squared Error)
- Hyperparameters: Learning rate $\eta = 0.3$, L2 regularization $\lambda = 1.0$, Minimum split gain $\gamma = 0.0$

Loss Function for sample $i$: $L(y_i, \hat{y}_i) = \frac{1}{2}(y_i - \hat{y}_i)^2$

1. **Base Prediction & Gradients/Hessians**
    Initial prediction $\hat{y}^{(0)}$ is set to the baseline mean:
    $$\hat{y}^{(0)} = \frac{10 + 20 + 25 + 35}{4} = 22.5$$

    For Mean Squared Error, the derivatives are:
    - Gradient ($g_i$): $g_i = \frac{\partial L}{\partial \hat{y}_i} = \hat{y}_i^{(0)} - y_i$
    - Hessian ($h_i$): $h_i = \frac{\partial^2 L}{\partial \hat{y}_i^2} = 1.0$

    |i|$x_i$|​$y_i$|​$\hat {y}_​i^{(0)}$|​Gradient $g_i​=\hat y_​i^{(0)}​−y_i$|​Hessian $h_i$|
    |---|---|---|---|---|---|
    |​1|1|10|22.5|$+12.5$|1.0|
    |2|2|20|22.5|$+2.5$|1.0|
    |3|3|25|22.5|$-2.5$|1.0|
    |4|4|35|22.5|$-12.5$|1.0|

    $$\sum g_i = +12.5 + 2.5 - 2.5 - 12.5 = 0.0, \quad \sum h_i = 1.0 + 1.0 + 1.0 + 1.0 = 4.0$$

2. **Calculate Root Node Similarity Score**
$$\text{Similarity}_{\text{Root}} = \frac{\left(\sum g_i\right)^2}{\sum h_i + \lambda} = \frac{(0.0)^2}{4.0 + 1.0} = \mathbf{0.0}$$
    

3. **Evaluate Candidate Splits**
    Evaluating candidate split thresholds on $X$:
    
    **Split Candidate 1: $X \le 1.5$**
    
    - Left Leaf ($x \in \{1\}$): $g_L = +12.5$, $h_L = 1.0$
    $$\text{Similarity}_L = \frac{(12.5)^2}{1.0 + 1.0} = \frac{156.25}{2.0} = \mathbf{78.125}$$
    
    - Right Leaf ($x \in \{2, 3, 4\}$): $g_R = +2.5 - 2.5 - 12.5 = -12.5$, $h_R = 3.0$
    $$\text{Similarity}_R = \frac{(-12.5)^2}{3.0 + 1.0} = \frac{156.25}{4.0} = \mathbf{39.0625}$$
    
    - Gain:
    $$\text{Gain} = 78.125 + 39.0625 - 0.0 - 0.0 = \mathbf{117.1875}$$
    
   **Split Candidate 2: $X \le 2.5$ (Optimal)**
   
   - Left Leaf ($x \in \{1, 2\}$): $g_L = +12.5 + 2.5 = +15.0$, $h_L = 2.0$
   $$\text{Similarity}_L = \frac{(+15.0)^2}{2.0 + 1.0} = \frac{225.0}{3.0} = \mathbf{75.0}$$
   
   - Right Leaf ($x \in \{3, 4\}$): $g_R = -2.5 - 12.5 = -15.0$, $h_R = 2.0$
   $$\text{Similarity}_R = \frac{(-15.0)^2}{2.0 + 1.0} = \frac{225.0}{3.0} = \mathbf{75.0}$$
   
   - Gain:
   $$\text{Gain} = 75.0 + 75.0 - 0.0 - 0.0 = \mathbf{150.0}$$
   
   **Split Candidate 3: $X \le 3.5$**
   
   - Left Leaf ($x \in \{1, 2, 3\}$): $g_L = +12.5$, $h_L = 3.0$
    $$\text{Similarity}_L = \frac{(+12.5)^2}{3.0 + 1.0} = \mathbf{39.0625}$$

    - Right Leaf ($x \in \{4\}$): $g_R = -12.5$, $h_R = 1.0$
    $$\text{Similarity}_R = \frac{(12.5)^2}{1.0 + 1.0} = \mathbf{78.125}$$

    - Gain: $\mathbf{117.1875}$

    The winning split is $X \le 2.5$ with maximum Gain of 150.0.

4. **Compute Optimal Leaf Output Weights ($w_j$)**
    
    $$w_j = -\frac{\sum g_i}{\sum h_i + \lambda}$$
    - Left Leaf Weight ($w_L$ for $x \le 2.5$):
    $$w_L = -\frac{+15.0}{2.0 + 1.0} = -\frac{15.0}{3.0} = \mathbf{-5.0}$$
    
    - Right Leaf Weight ($w_R$ for $x > 2.5$):
    $$w_R = -\frac{-15.0}{2.0 + 1.0} = -\frac{-15.0}{3.0} = \mathbf{+5.0}$$
    
5. **Update Predictions $\hat{y}^{(1)}$**
    Using learning rate $\eta = 0.3$:
    $$\hat{y}_i^{(1)} = \hat{y}_i^{(0)} + \eta \cdot w(x_i) = 22.5 + 0.3 \cdot w(x_i)$$
    
    - For $x_1 = 1, x_2 = 2$ ($x \le 2.5$):
    $$\hat{y}^{(1)} = 22.5 + 0.3(-5.0) = 22.5 - 1.5 = \mathbf{21.0}$$
    
    - For $x_3 = 3, x_4 = 4$ ($x > 2.5$):
    $$\hat{y}^{(1)} = 22.5 + 0.3(+5.0) = 22.5 + 1.5 = \mathbf{24.0}$$
    
**Comparison of Step 1 Updates**

|$x_i$|​True Target $y_i​$|Base Prediction $y^{​(0)}$|Updated Prediction $y^{​(1)}$|Direction|
|---|---|---|---|---|
|1|10|22.5|21.0 (or moving down relative to $x \in \{3,4\}$)|Shifted DOWN toward target ($10$)|
|2|20|22.5|21.0|Shifted DOWN toward target ($20$)|
|3|25|22.5|24.0|Shifted UP toward target ($25$)|
|4|35|22.5|24.0|Shifted UP toward target ($35$)|

Repeat the same steps again 

## Python Implementation

In [9]:
import numpy as np


class XGBoostTree:
    """Single XGBoost Decision Tree built using Gradients, Hessians, and Regularisation."""

    def __init__(self, reg_lambda=1.0, min_child_weight=1.0, max_depth=2, gamma=0.0):
        self.reg_lambda = reg_lambda
        self.min_child_weight = min_child_weight
        self.max_depth = max_depth
        self.gamma = gamma
        self.tree = None

    def _calc_similarity(self, g, h):
        """Similarity Score = (sum g)^2 / (sum h + lambda)"""
        return (np.sum(g) ** 2) / (np.sum(h) + self.reg_lambda)

    def _calc_leaf_weight(self, g, h):
        """Optimal Leaf Weight w* = - sum(g) / (sum(h) + lambda)"""
        return -np.sum(g) / (np.sum(h) + self.reg_lambda)

    def _build_tree(self, X, g, h, depth=0):
        n_samples, n_features = X.shape
        sum_h = np.sum(h)

        # Base case: Max depth reached or insufficient hessian sum (min_child_weight)
        if depth >= self.max_depth or sum_h < self.min_child_weight:
            return {"leaf": True, "weight": self._calc_leaf_weight(g, h)}

        best_gain = 0.0
        best_split = None
        root_sim = self._calc_similarity(g, h)

        for f_idx in range(n_features):
            X_col = X[:, f_idx]
            sorted_unique = np.sort(np.unique(X_col))

            if len(sorted_unique) <= 1:
                continue

            # Candidate split thresholds at midpoints
            thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

            for t in thresholds:
                left_mask = X_col <= t
                right_mask = ~left_mask

                if not np.any(left_mask) or not np.any(right_mask):
                    continue

                g_L, h_L = g[left_mask], h[left_mask]
                g_R, h_R = g[right_mask], h[right_mask]

                # Minimum child weight constraint check
                if np.sum(h_L) < self.min_child_weight or np.sum(h_R) < self.min_child_weight:
                    continue

                sim_L = self._calc_similarity(g_L, h_L)
                sim_R = self._calc_similarity(g_R, h_R)

                gain = sim_L + sim_R - root_sim - self.gamma

                if gain > best_gain:
                    best_gain = gain
                    best_split = (f_idx, t, left_mask, right_mask)

        # If no positive gain achieved, convert node to leaf (built-in pruning)
        if best_split is None or best_gain <= 0:
            return {"leaf": True, "weight": self._calc_leaf_weight(g, h)}

        f_idx, t, left_mask, right_mask = best_split

        return {
            "leaf": False,
            "feature_idx": f_idx,
            "threshold": t,
            "left": self._build_tree(X[left_mask], g[left_mask], h[left_mask], depth + 1),
            "right": self._build_tree(X[right_mask], g[right_mask], h[right_mask], depth + 1),
        }

    def fit(self, X, g, h):
        self.tree = self._build_tree(X, g, h, depth=0)

    def _predict_sample(self, x, node):
        if node["leaf"]:
            return node["weight"]
        if x[node["feature_idx"]] <= node["threshold"]:
            return self._predict_sample(x, node["left"])
        return self._predict_sample(x, node["right"])

    def predict(self, X):
        return np.array([self._predict_sample(x, self.tree) for x in X])


class XGBoostRegressor:
    """Extreme Gradient Boosting Regressor (MSE Loss)."""

    def __init__(
        self,
        n_estimators=1,
        learning_rate=0.3,
        reg_lambda=1.0,
        min_child_weight=1.0,
        max_depth=1,
        gamma=0.0,
    ):
        self.M = n_estimators
        self.eta = learning_rate
        self.reg_lambda = reg_lambda
        self.min_child_weight = min_child_weight
        self.max_depth = max_depth
        self.gamma = gamma
        self.trees = []
        self.F0 = 0.0

    def fit(self, X, y):
        # Step 1: Base guess F0 (Mean of targets for MSE)
        self.F0 = np.mean(y)
        F = np.full_like(y, self.F0, dtype=np.float64)

        print("--- XGBOOST TRAINING LOOP ---")
        print(f"Base Prediction F0 = {self.F0:.3f}\n")

        for m in range(self.M):
            # Step 2: Compute Gradients (g) and Hessians (h) for MSE Loss
            # Loss L = 0.5 * (y - F)^2  =>  g = F - y,  h = 1.0
            g = F - y
            h = np.ones_like(y, dtype=np.float64)

            # Step 3 & 4: Build tree using Similarity Score and Gain
            tree = XGBoostTree(
                reg_lambda=self.reg_lambda,
                min_child_weight=self.min_child_weight,
                max_depth=self.max_depth,
                gamma=self.gamma,
            )
            tree.fit(X, g, h)

            # Step 5 & 6: Compute predictions & update ensemble log-odds with shrinkage (eta)
            w = tree.predict(X)
            F += self.eta * w
            self.trees.append(tree)

            print(f"Round {m+1} Predictions: {np.round(F, 3)}")

    def predict(self, X):
        F = np.full(X.shape[0], self.F0, dtype=np.float64)
        for tree in self.trees:
            F += self.eta * tree.predict(X)
        return F


if __name__ == "__main__":
    # Toy dataset from our manual math verification
    X_train = np.array([[1.0], [2.0], [3.0], [4.0]])
    y_train = np.array([10.0, 20.0, 25.0, 35.0])

    xgb = XGBoostRegressor(
        n_estimators=50,
        learning_rate=0.3,
        reg_lambda=1.0,
        max_depth=1,
        gamma=0.0,
    )
    xgb.fit(X_train, y_train)

    preds = xgb.predict(X_train)

    print("\n--- INFERENCE RESULTS ---")
    for x_val, y_true, y_pred in zip(X_train.ravel(), y_train, preds):
        print(f"X = {x_val}: True = {y_true} -> Model Prediction = {y_pred:.3f}")

--- XGBOOST TRAINING LOOP ---
Base Prediction F0 = 22.500

Round 1 Predictions: [21. 21. 24. 24.]
Round 2 Predictions: [19.8 19.8 25.2 25.2]
Round 3 Predictions: [18.33  20.535 25.935 25.935]
Round 4 Predictions: [17.595 19.8   25.2   27.295]
Round 5 Predictions: [17.025 19.23  24.63  28.451]
Round 6 Predictions: [15.972 19.807 25.207 29.027]
Round 7 Predictions: [15.523 19.358 24.758 29.923]
Round 8 Predictions: [14.694 19.805 25.205 30.37 ]
Round 9 Predictions: [13.99  20.152 25.552 30.717]
Round 10 Predictions: [13.638 19.8   25.2   31.359]
Round 11 Predictions: [13.365 19.527 24.927 31.905]
Round 12 Predictions: [12.861 19.8   25.2   32.178]
Round 13 Predictions: [12.431 20.012 25.412 32.39 ]
Round 14 Predictions: [12.217 19.797 25.197 32.781]
Round 15 Predictions: [11.885 19.964 25.364 32.948]
Round 16 Predictions: [11.719 19.798 25.198 33.256]
Round 17 Predictions: [11.59  19.67  25.07  33.518]
Round 18 Predictions: [11.352 19.8   25.2   33.648]
Round 19 Predictions: [11.25  19.6

# XGBOOST (Classifier)

**Formula Comparison Table**

|Component|XGBoost Regressor (MSE Loss)|XGBoost Classifier (Binary Log-Loss)|
|---|---|---|
|Loss Function $L(y, \hat{y})$ |$\frac{1}{2}(y_i - \hat{y}_i)^2$|$-y_i \ln(p_i) - (1 - y_i) \ln(1 - p_i)$|
|Prediction Space ($\hat{y}$ or $F$)|Continuous real value $\hat{y} \in (-\infty, +\infty)$|Raw Log-Odds score $F \in (-\infty, +\infty)$|
|Probability Transformation $p_i$|N/A (Directly outputs target value)|$p_i = \sigma(F_i) = \frac{1}{1 + e^{-F_i}}$|
|Base Prediction $F_0$|$F_0 = \bar{y} = \frac{1}{N}\sum y_i$ (Mean)|$F_0 = \ln\left(\frac{\sum y_i}{N - \sum y_i}\right)$ (Log-Odds)|
|Gradient ($g_i = \frac{\partial L}{\partial F_i}$)|$g_i = \hat{y}_i - y_i$|$g_i = p_i - y_i$|
|Hessian ($h_i = \frac{\partial^2 L}{\partial F_i^2}$)$|h_i = 1.0$ (Constant curvature)|$h_i = p_i(1 - p_i)$ (Dynamic curvature)|
|Leaf Weight ($w_j^*$)|$w_j^* = -\frac{\sum g_i}{\sum h_i + \lambda} = -\frac{\sum (\hat{y}_i - y_i)}{N_j + \lambda}$|$w_j^* = -\frac{\sum g_i}{\sum h_i + \lambda} = -\frac{\sum (p_i - y_i)}{\sum p_i(1 - p_i) + \lambda}$|
|Final Inference Prediction|$\hat{y}_{\text{final}} = F_0 + \eta \sum_{m=1}^M w_m(x)$|$p_{\text{final}} = \sigma\left( F_0 + \eta \sum_{m=1}^M w_m(x) \right)$|

## Python Implementation

In [10]:
import numpy as np


def sigmoid(z):
    """Numerically stable Sigmoid function to map log-odds to probabilities."""
    return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))


class XGBoostTree:
    """Single XGBoost Decision Tree for Classification using Binary Log-Loss."""

    def __init__(self, reg_lambda=1.0, min_child_weight=1.0, max_depth=2, gamma=0.0):
        self.reg_lambda = reg_lambda
        self.min_child_weight = min_child_weight
        self.max_depth = max_depth
        self.gamma = gamma
        self.tree = None

    def _calc_similarity(self, g, h):
        """Similarity Score = (sum g)^2 / (sum h + lambda)"""
        return (np.sum(g) ** 2) / (np.sum(h) + self.reg_lambda)

    def _calc_leaf_weight(self, g, h):
        """Optimal Leaf Weight w* = - sum(g) / (sum(h) + lambda)"""
        return -np.sum(g) / (np.sum(h) + self.reg_lambda)

    def _build_tree(self, X, g, h, depth=0):
        n_samples, n_features = X.shape
        sum_h = np.sum(h)

        # Base case: Max depth reached or total sum of Hessians below threshold
        if depth >= self.max_depth or sum_h < self.min_child_weight:
            return {"leaf": True, "weight": self._calc_leaf_weight(g, h)}

        best_gain = 0.0
        best_split = None
        root_sim = self._calc_similarity(g, h)

        for f_idx in range(n_features):
            X_col = X[:, f_idx]
            sorted_unique = np.sort(np.unique(X_col))

            if len(sorted_unique) <= 1:
                continue

            # Evaluate candidate split thresholds at midpoints
            thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

            for t in thresholds:
                left_mask = X_col <= t
                right_mask = ~left_mask

                if not np.any(left_mask) or not np.any(right_mask):
                    continue

                g_L, h_L = g[left_mask], h[left_mask]
                g_R, h_R = g[right_mask], h[right_mask]

                # Minimum child weight constraint check (uses sum of Hessians)
                if np.sum(h_L) < self.min_child_weight or np.sum(h_R) < self.min_child_weight:
                    continue

                sim_L = self._calc_similarity(g_L, h_L)
                sim_R = self._calc_similarity(g_R, h_R)

                gain = sim_L + sim_R - root_sim - self.gamma

                if gain > best_gain:
                    best_gain = gain
                    best_split = (f_idx, t, left_mask, right_mask)

        # Built-in pruning: convert node to leaf if Gain <= 0
        if best_split is None or best_gain <= 0:
            return {"leaf": True, "weight": self._calc_leaf_weight(g, h)}

        f_idx, t, left_mask, right_mask = best_split

        return {
            "leaf": False,
            "feature_idx": f_idx,
            "threshold": t,
            "left": self._build_tree(X[left_mask], g[left_mask], h[left_mask], depth + 1),
            "right": self._build_tree(X[right_mask], g[right_mask], h[right_mask], depth + 1),
        }

    def fit(self, X, g, h):
        self.tree = self._build_tree(X, g, h, depth=0)

    def _predict_sample(self, x, node):
        if node["leaf"]:
            return node["weight"]
        if x[node["feature_idx"]] <= node["threshold"]:
            return self._predict_sample(x, node["left"])
        return self._predict_sample(x, node["right"])

    def predict(self, X):
        return np.array([self._predict_sample(x, self.tree) for x in X])


class XGBoostClassifier:
    """Extreme Gradient Boosting Classifier (Binary Log-Loss)."""

    def __init__(
        self,
        n_estimators=1,
        learning_rate=0.5,
        reg_lambda=1.0,
        min_child_weight=0.1,
        max_depth=1,
        gamma=0.0,
    ):
        self.M = n_estimators
        self.eta = learning_rate
        self.reg_lambda = reg_lambda
        self.min_child_weight = min_child_weight
        self.max_depth = max_depth
        self.gamma = gamma
        self.trees = []
        self.F0 = 0.0

    def fit(self, X, y):
        n_samples = X.shape[0]

        # Step 1: Base guess F0 (Log-odds of positive class)
        pos_count = np.sum(y == 1)
        neg_count = np.sum(y == 0)
        self.F0 = np.log(pos_count / neg_count)

        # Initialize raw log-odds F
        F = np.full(n_samples, self.F0, dtype=np.float64)

        print("--- XGBOOST CLASSIFIER TRAINING LOOP ---")
        print(f"Base Log-Odds F0 = {self.F0:.3f}")
        print(f"Base Probability p0 = {sigmoid(self.F0):.3f}\n")

        for m in range(self.M):
            # Step 2: Calculate probabilities, gradients (g), and hessians (h)
            p = sigmoid(F)
            g = p - y
            h = p * (1.0 - p)

            # Step 3 & 4: Build classification stump/tree
            tree = XGBoostTree(
                reg_lambda=self.reg_lambda,
                min_child_weight=self.min_child_weight,
                max_depth=self.max_depth,
                gamma=self.gamma,
            )
            tree.fit(X, g, h)

            # Step 5 & 6: Update log-odds predictions using shrinkage (eta)
            w = tree.predict(X)
            F += self.eta * w
            self.trees.append(tree)

            print(f"Round {m+1} Log-Odds F: {np.round(F, 3)}")
            print(f"Round {m+1} Probabilities: {np.round(sigmoid(F), 3)}\n")

    def predict_proba(self, X):
        """Returns predicted class probabilities."""
        F = np.full(X.shape[0], self.F0, dtype=np.float64)
        for tree in self.trees:
            F += self.eta * tree.predict(X)
        return sigmoid(F)

    def predict(self, X, threshold=0.5):
        """Returns binary class labels (0 or 1)."""
        return (self.predict_proba(X) >= threshold).astype(int)


if __name__ == "__main__":
    # Dataset matching our manual classification math calculation
    X_train = np.array([[1.0], [2.0], [3.0], [4.0]])
    y_train = np.array([0, 0, 1, 1])

    xgb_clf = XGBoostClassifier(
        n_estimators=1,
        learning_rate=0.5,
        reg_lambda=1.0,
        min_child_weight=0.1,
        max_depth=1,
        gamma=0.0,
    )
    xgb_clf.fit(X_train, y_train)

    probs = xgb_clf.predict_proba(X_train)
    preds = xgb_clf.predict(X_train)

    print("--- INFERENCE RESULTS ---")
    for x_val, y_true, p_val, c_val in zip(X_train.ravel(), y_train, probs, preds):
        print(
            f"X = {x_val}: True = {y_true} -> Prob = {p_val:.3f} | Predicted Class = {c_val}"
        )

--- XGBOOST CLASSIFIER TRAINING LOOP ---
Base Log-Odds F0 = 0.000
Base Probability p0 = 0.500

Round 1 Log-Odds F: [-0.333 -0.333  0.333  0.333]
Round 1 Probabilities: [0.417 0.417 0.583 0.583]

--- INFERENCE RESULTS ---
X = 1.0: True = 0 -> Prob = 0.417 | Predicted Class = 0
X = 2.0: True = 0 -> Prob = 0.417 | Predicted Class = 0
X = 3.0: True = 1 -> Prob = 0.583 | Predicted Class = 1
X = 4.0: True = 1 -> Prob = 0.583 | Predicted Class = 1
